# Scripts of the World

How are writing systems distributed across languages? This notebook maps
ISO 15924 scripts to their languages and weights them by global speaker counts.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [ ]:
%pip install -q pandas plotly matplotlib


In [ ]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [ ]:
db = low.LanguagesOfTheWorld()
print(db)
print(f"Writing systems: {len(db.scripts)}")

## 2 · Lookup scripts and languages

In [ ]:
deva = db.scripts.get("deva")
print(f"{deva.label} ({deva.code}): {len(deva.languages)} languages")

hin = db.languages.get("hin")
print(f"\n{hin.label} scripts:")
for s in hin.scripts:
    primary = " (primary)" if s == hin.primary_script else ""
    print(f"  {s.label} ({s.code}){primary}")

print(f"\nScripts via collection helper:")
for s in db.scripts.for_language("hin"):
    print(f"  {s.label}")

## 3 · Languages per script

In [ ]:
script_rows = [
    {
        "code": script.code,
        "script": script.label,
        "language_count": len(script.languages),
        "speaker_total": sum(l.speaker_count or 0 for l in script.languages),
    }
    for script in db.scripts
]

script_df = pd.DataFrame(script_rows).sort_values("language_count", ascending=False)
script_df.head(15)

## 4 · Bar chart — languages per script (top 15)

In [ ]:
top15 = script_df.head(15)

fig = px.bar(
    top15,
    x="language_count",
    y="script",
    orientation="h",
    text="language_count",
    labels={"language_count": "Languages", "script": ""},
    title="Top 15 Writing Systems by Number of Languages",
    color_discrete_sequence=["#4c78a8"],
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig.update_traces(textposition="outside")
fig.show()

## 5 · Treemap — scripts weighted by speakers

In [ ]:
weighted = script_df[script_df["speaker_total"] > 0].head(20)

fig2 = px.treemap(
    weighted,
    path=["script"],
    values="speaker_total",
    title="Top 20 Scripts by Total Speaker Count",
    color="speaker_total",
    color_continuous_scale="Teal",
)
fig2.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig2.show()

## 6 · Languages with multiple scripts

In [ ]:
multi = [
    {
        "language": lang.label,
        "part3": lang.part3,
        "scripts": ", ".join(s.label for s in lang.scripts),
        "script_count": len(lang.scripts),
        "primary": lang.primary_script.label if lang.primary_script else None,
    }
    for lang in db.languages
    if len(lang.scripts) > 1
]

multi_df = pd.DataFrame(multi).sort_values("script_count", ascending=False)
print(f"Languages with multiple scripts: {len(multi_df)}")
multi_df.head(15)

## 7 · Summary

In [ ]:
no_script = sum(1 for l in db.languages if not l.scripts)
print(f"Languages without a script record: {no_script:,}")
print(f"Most widespread script: {script_df.iloc[0]['script']} ({script_df.iloc[0]['language_count']} languages)")
print(f"Highest speaker total: {script_df.sort_values('speaker_total', ascending=False).iloc[0]['script']}")